In [1]:
# ==========================================================
# Notebook 06: Cross-Encoder Re-ranking
# ==========================================================

import re

import faiss
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df["search_text"] = enterprise_df["title"] + " " + enterprise_df["content"]

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team,search_text
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering,Phoenix Deployment Discussion The Phoenix-2026...
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance,Model Training Update The semantic search mode...
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR,Phoenix Architecture Wiki Project Phoenix-2026...
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering,Enterprise Search Design Hybrid search combine...
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering,PHX-245 Database Migration Complete PostgreSQL...


In [3]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)

    return text.split()

In [4]:
tokenized_corpus = [preprocess_text(doc) for doc in enterprise_df["search_text"]]

bm25 = BM25Okapi(tokenized_corpus)

In [5]:
# Bi-Encoder (retrieval)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Cross-Encoder (re-ranking)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
document_embeddings = embedding_model.encode(
    enterprise_df["search_text"].tolist(), convert_to_numpy=True, show_progress_bar=True
)

document_embeddings = np.array(document_embeddings).astype("float32")

faiss.normalize_L2(document_embeddings)

faiss_index = faiss.IndexFlatIP(document_embeddings.shape[1])

faiss_index.add(document_embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
def retrieve_candidates(query, top_k=20):

    # BM25
    bm25_scores = bm25.get_scores(preprocess_text(query))

    bm25_rank = np.argsort(bm25_scores)[::-1][:top_k]

    # Semantic
    query_embedding = embedding_model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    _, semantic_rank = faiss_index.search(query_embedding, top_k)

    semantic_rank = list(semantic_rank[0])

    # Merge candidates
    candidates = list(set(list(bm25_rank) + semantic_rank))

    return candidates

In [8]:
def rerank_documents(query, candidate_indices, dataframe, cross_encoder_model, top_k=5):

    pairs = [(query, dataframe.iloc[idx]["search_text"]) for idx in candidate_indices]

    scores = cross_encoder_model.predict(pairs)

    results = []

    for idx, score in zip(candidate_indices, scores):

        row = dataframe.iloc[idx].to_dict()

        row["cross_encoder_score"] = float(score)

        results.append(row)

    results = sorted(results, key=lambda x: x["cross_encoder_score"], reverse=True)

    return pd.DataFrame(results[:top_k])

In [9]:
query = "vector database for enterprise search"

candidate_docs = retrieve_candidates(query, top_k=10)

reranked_results = rerank_documents(
    query, candidate_docs, enterprise_df, cross_encoder, top_k=5
)

reranked_results[["doc_id", "title", "cross_encoder_score"]]

,doc_id,title,cross_encoder_score
0,DOC0004,Enterprise Search Design,6.251303
1,DOC0006,SEC-104 ACL Enhancement,-0.970574
2,DOC0006,SEC-104 ACL Enhancement,-0.970574
3,DOC0002,Model Training Update,-11.020158
4,DOC0005,PHX-245 Database Migration,-11.299570


In [10]:
query = "vector database for enterprise search"

print("=" * 70)
print("HYBRID CANDIDATES")
print("=" * 70)

candidate_docs = retrieve_candidates(query, top_k=5)

print(enterprise_df.iloc[candidate_docs][["title"]])

print("\n")

print("=" * 70)
print("CROSS-ENCODER RE-RANKED")
print("=" * 70)

print(
    rerank_documents(query, candidate_docs, enterprise_df, cross_encoder, top_k=5)[
        ["title", "cross_encoder_score"]
    ]
)

HYBRID CANDIDATES
                           title
0  Phoenix Deployment Discussion
1          Model Training Update
2      Phoenix Architecture Wiki
3       Enterprise Search Design
4     PHX-245 Database Migration
5        SEC-104 ACL Enhancement


CROSS-ENCODER RE-RANKED
                           title  cross_encoder_score
0       Enterprise Search Design             6.251303
1        SEC-104 ACL Enhancement            -0.970574
2          Model Training Update           -11.020158
3     PHX-245 Database Migration           -11.299570
4  Phoenix Deployment Discussion           -11.401681


In [11]:
import os
import pickle

OUTPUT_PATH = "artifacts/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

config = {
    "bi_encoder": "sentence-transformers/all-MiniLM-L6-v2",
    "cross_encoder": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "candidate_pool_size": 20,
}

with open(OUTPUT_PATH + "cross_encoder_config.pkl", "wb") as file:

    pickle.dump(config, file)

print("Cross-Encoder configuration saved!")

Cross-Encoder configuration saved!
